# Experimental Notebook - Topology scAGCL Model

Source code of `scAGCL` can be found in [levinhcntt/scAGCL](https://github.com/levinhcntt/scAGCL)

## Experiment configuration

In [ ]:
SEEDS = [11111, 22222, 33333, 44444, 55555]
DATASETS = [
    "QX_Lung",
    "QX_Marrow",
    "QS_Heart",
    "QS_Lung",
    "QS_Diaphragm",
    "QS_Limb_Muscle",
]

from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "topology_signature.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel from inside the TopoCLR repository.")

BASE_DIR = PROJECT_ROOT / "data"  # Folder containing the local .h5 files.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Total epochs per mode and topology-enabled epochs within that total.
num_epochs = 500
topology_epochs = 250
MAX_TOPOLOGY_DIM = 1
distance_type = "similarity"
loss_type = "MSE"  # MSE, Log-Ratio, Cosine-Log, or Topo-KL
TOPO_WEIGHT = 20

# Model and optimizer
subgraph_size = 256
num_proj_hidden = 256
num_hidden = 256
num_layers = 2
tau = 0.5
learning_rate = 0.001
weight_decay = 1e-5

# Graph augmentation
num_itersAdv = 10
lam = 1
alpha = 100
beta = 0.01
edge_r1 = 0.4
edge_r2 = 0.3
feature_r1 = 0.3
feature_r2 = 0.4

RUN_ROBUSTNESS = False
ROBUSTNESS_DATASET = "QS_Diaphragm"  # Used only when RUN_ROBUSTNESS is True.

## About this notebook

In this notebook, we rerun the experiments introduced in paper "scAGCL: Deep clustering of single-cell RNA-seq using Adversarial Graph Contrastive Learning" and compare with the new version of adding topology-refinement between two augmented views and two adversarial views.

## Load library

In [ ]:
import os
import sys
import h5py
import numpy as np
import torch
import random
from torch_geometric.data import Data
from scipy.spatial.distance import pdist, squareform
from sklearn.neighbors import NearestNeighbors
import networkx as nx
import pandas as pd

from IPython.display import display

def load_expression(path):
    with h5py.File(path, "r") as data_file:
        counts = np.asarray(data_file["X"])
        labels = np.asarray(data_file["Y"]).reshape(-1)
    if counts.ndim != 2 or counts.shape[0] != labels.size or not labels.size:
        raise ValueError("Expected a count matrix and one label per cell.")
    return counts, labels


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. Module Code

## 1.1 Preprocessing

In [ ]:
import scanpy as sc
from utils.preprocessing import normalize
from sklearn.preprocessing import LabelEncoder

def normalization(features_):
    features = features_.copy()
    for i in range(len(features)):
        features[i] = features[i] / sum(features[i]) * 100000
    features = np.log2(features + 1)
    return features

def preprocess(X, nb_genes=2000, *, labels=None):
    """
    Preprocessing phase as proposed in scanpy package. Keeps only nb_genes most variable genes and normalizes the data to 0 mean and 1 std.
    """

    print("There are " + str(X.shape[0]) + " cells, " + str(X.shape[1]) + " genes")
    print("Data Preprocessing!")
    
    adata = sc.AnnData(X)
    if labels is not None:
        adata.obs["reference_label"] = labels

    adata = normalize(adata,
                      copy=True,
                      highly_genes=nb_genes,
                      size_factors=True,
                      normalize_input=True,
                      logtrans_input=True)
    X = adata.X.astype(np.float32)
    print(f"Keeping {X.shape[1]} genes")
    if labels is not None:
        return X, adata.obs["reference_label"].to_numpy()
    return X

def load_h5_data1(data_path):
    print("Reading data!")
    counts, labels = load_expression(data_path)
    labels = LabelEncoder().fit_transform(labels)
    return preprocess(counts.astype(np.float32), nb_genes=2000, labels=labels)


def GraphConstruction(data, features, num_clusters):
    cell_num = features.shape[0]
    average_num = cell_num // num_clusters
    neighbor_num = average_num // 10
    neighbor_num = min(neighbor_num, 15)
    neighbor_num = max(neighbor_num, 5)
 
    # Calculate Pearson distance matrix
    dis_matrix = squareform(pdist(features, metric='correlation'))
    # Build kNN graph
    nbrs = NearestNeighbors(n_neighbors=neighbor_num, metric='precomputed').fit(dis_matrix)
    _, indices = nbrs.kneighbors(dis_matrix)  # Get only indices
 
    # Create adjacency matrix
    n_samples = features.shape[0]
    adj_matrix = np.zeros((n_samples, n_samples))

    for i in range(n_samples):
        for j in indices[i]:
             adj_matrix[i, j] = 1  

      
    #Create a list of egdes
    edge_list = torch.empty((2,0), dtype=torch.int64)
    for i in range( adj_matrix.shape[0]):
     for j in range(i + 1,   adj_matrix.shape[1]):
        if  adj_matrix[i, j] == 1:
            col = torch.tensor([i, j], dtype=torch.int64)
            edge_list = torch.cat((edge_list, col.unsqueeze(1)), dim=1)
    data.edge_index=edge_list
          
    #generate a graph
    G = nx.from_numpy_array(adj_matrix)
    print("Building a " + str(G))
    print("===================================================")

    return G

## 1.2 Model Architecture

In [ ]:
from torch import nn
from torch.nn import Module, Parameter
import math
import torch.nn.functional as F

class GCNConv(Module):
    def __init__(self, in_features, out_features, bias=True):
        super(GCNConv, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)

    def forward(self, input, adj):
        support = torch.mm(input, self.weight)
        output = torch.spmm(adj, support)
        if self.bias is not None:
            return output + self.bias
        else:
            return output

    def __repr__(self):
        return self.__class__.__name__ + ' (' \
               + str(self.in_features) + ' -> ' \
               + str(self.out_features) + ')'

class Encoder(nn.Module):
    def __init__(self, dim_in: int, dim_out: int, num_layers: int = 2):
        super(Encoder, self).__init__()
        self.base_model = GCNConv
        self.activation = F.relu
        assert num_layers >= 2
        self.num_layers= num_layers
        self.conv = [GCNConv(dim_in, 2 * dim_out)]
        for _ in range(1, num_layers-1):
            self.conv.append(GCNConv(2 * dim_out, 2 * dim_out))
        self.conv.append(GCNConv(2 * dim_out, dim_out))
        self.conv = nn.ModuleList(self.conv)

        self.dim_in = dim_in
        self.dim_out = dim_out


    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        for i in range(self.num_layers):
            x = self.activation(self.conv[i](x, edge_index))
        return x
    
class AGCLModel(torch.nn.Module):
    def __init__(self, encoder: Encoder, n_hidden: int, n_proj_hidden: int,
                 tau: float = 0.5):
        super(AGCLModel, self).__init__()
        self.encoder: Encoder = encoder
        self.tau: float = tau

        self.fc_layer1 = torch.nn.Linear(n_hidden, n_proj_hidden)
        self.fc_layer2 = torch.nn.Linear(n_proj_hidden, n_hidden)
        
    def forward(self, x: torch.Tensor,
                Adj: torch.Tensor) -> torch.Tensor:
       
        return self.encoder(x, Adj)

    def projection(self, z: torch.Tensor) -> torch.Tensor:
        z = F.elu(self.fc_layer1(z))
        return self.fc_layer2(z)

    #Codes are modified from https://github.com/Shengyu-Feng/ARIEL
    def sim(self, z1: torch.Tensor, z2: torch.Tensor):
        z1 = F.normalize(z1)
        z2 = F.normalize(z2)
        return torch.mm(z1, z2.t())

    def semi_loss(self, z1: torch.Tensor, z2: torch.Tensor):
        f = lambda x: torch.exp(x / self.tau)
        refl_sim = f(self.sim(z1, z1))
        between_sim = f(self.sim(z1, z2))

        return -torch.log(
            between_sim.diag()
            / (refl_sim.sum(1) + between_sim.sum(1) - refl_sim.diag()))

    def batched_semi_loss(self, z1: torch.Tensor, z2: torch.Tensor,
                          batch_size: int):
        # Space complexity: O(BN) (semi_loss: O(N^2))
        device = z1.device
        num_nodes = z1.size(0)
        num_batches = (num_nodes - 1) // batch_size + 1
        f = lambda x: torch.exp(x / self.tau)
        indices = torch.arange(0, num_nodes).to(device)
        losses = []

        for i in range(num_batches):
            mask = indices[i * batch_size:(i + 1) * batch_size]
            refl_sim = f(self.sim(z1[mask], z1))  # [B, N]
            between_sim = f(self.sim(z1[mask], z2))  # [B, N]

            losses.append(-torch.log(
                between_sim[:, i * batch_size:(i + 1) * batch_size].diag()
                / (refl_sim.sum(1) + between_sim.sum(1)
                   - refl_sim[:, i * batch_size:(i + 1) * batch_size].diag())))

        return torch.cat(losses)

    def loss(self, z1: torch.Tensor, z2: torch.Tensor,
             mean: bool = True, batch_size: int = 0):
        h1 = self.projection(z1)
        h2 = self.projection(z2)
        
            
        if batch_size == 0:
            l1 = self.semi_loss(h1, h2)
            l2 = self.semi_loss(h2, h1)
        else:
            l1 = self.batched_semi_loss(h1, h2, batch_size)
            l2 = self.batched_semi_loss(h2, h1, batch_size)

        ret = (l1 + l2) * 0.5
        ret = ret.mean() 

        return ret

## 1.3 Adversarial Attack

In [ ]:
# from torch_scatter import scatter_add
from torch_geometric.utils import add_self_loops
from torch import Tensor
from typing import Tuple

def normalize_adj_tensor(adj):
    """Symmetrically normalize adjacency tensor."""
    rowsum = torch.sum(adj,1)
    d_inv_sqrt = torch.pow(rowsum, -0.5)
    d_inv_sqrt[d_inv_sqrt == float("Inf")] = 0.
    d_mat_inv_sqrt = torch.diag(d_inv_sqrt)
    return torch.mm(torch.mm(adj,d_mat_inv_sqrt).transpose(0,1),d_mat_inv_sqrt)

def normalize_adj_tensor_sp(adj):
    """Symmetrically normalize sparse adjacency tensor."""
    device = adj.device
    adj = adj.to("cpu")
    rowsum = torch.spmm(adj, torch.ones((adj.size(0),1))).reshape(-1)
    d_inv_sqrt = torch.pow(rowsum, -0.5)
    d_inv_sqrt[d_inv_sqrt == float("Inf")] = 0.
    d_mat_inv_sqrt = torch.diag(d_inv_sqrt)
    adj = torch.mm(torch.smm(adj.transpose(0,1),d_mat_inv_sqrt.transpose(0,1)),d_mat_inv_sqrt)
    return adj.to(device)

def edge2adj(x, edge_index):
    """Convert edge index to adjacency matrix"""
    num_nodes = x.shape[0]
    tmp, _ = add_self_loops(edge_index, num_nodes=num_nodes)
    edge_weight = torch.ones(tmp.size(1), dtype=None,
                                     device=edge_index.device)
    row, col = tmp[0], tmp[1]
    # deg = scatter_add(edge_weight, row, dim=0, dim_size=num_nodes)
    deg = torch.zeros(num_nodes, dtype=edge_weight.dtype, device=edge_weight.device)
    deg.scatter_add_(0, row, edge_weight)
    
    deg_inv_sqrt = deg.pow_(-0.5)
    deg_inv_sqrt.masked_fill_(deg_inv_sqrt == float('inf'), 0)
    edge_weight = deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]
    #return torch.sparse.FloatTensor(tmp, edge_weight,torch.Size((num_nodes, num_nodes)))
    return torch.sparse_coo_tensor(tmp, edge_weight, torch.Size((num_nodes, num_nodes)))

def EdgeDropping(edge_index: Tensor, p: float = 0.5,
                 force_undirected: bool = False,
                 training: bool = True) -> Tuple[Tensor, Tensor]:
    if p < 0. or p > 1.:
        raise ValueError(f'Dropout probability has to be between 0 and 1 '
                         f'(got {p}')

    if not training or p == 0.0:
        edge_mask = edge_index.new_ones(edge_index.size(1), dtype=torch.bool)
        return edge_index, edge_mask

    row, col = edge_index

    edge_mask = torch.bernoulli(torch.ones(row.size(0), device=edge_index.device) * (1 - p)).type(torch.bool)

    if force_undirected:
        edge_mask[row > col] = False

    edge_index = edge_index[:, edge_mask]

    if force_undirected:
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_mask = edge_mask.nonzero().repeat((2, 1)).squeeze()

    return edge_index, edge_mask


def GeneDropping(x, drop_prob):
    drop_mask = torch.bernoulli(torch.ones(x.size(1), device=x.device) * (1 - drop_prob)).type(torch.bool) 
    x = x.clone()
    x[:, drop_mask] = 0
    return x

def GraphAdversarialAttack(model, edge_index_1, edge_index_2, x_1, x_2, iters, node_ratio, alpha, beta):
    """ PGD attack on both features and edges"""

    for param in  model.parameters():
        param.requires_grad = False
    
    model.eval()
    device = x_1.device
    total_edges = edge_index_2.shape[1]         
    n_node = x_2.shape[0]
    eps = total_edges * node_ratio/2
    xi = 1e-3
    
    A_ = torch.sparse_coo_tensor(edge_index_2, torch.ones(total_edges,device=device), torch.Size((n_node, n_node))).to_dense() 
    C_ = torch.ones_like(A_) - 2 * A_ - torch.eye(A_.shape[0],device=device)
    S_ = torch.zeros_like(A_, requires_grad= True)
    mask = torch.ones_like(A_)
    mask = mask - torch.tril(mask)
    delta = torch.zeros_like(x_2, device=device, requires_grad=True)
    adj_1 = edge2adj(x_1, edge_index_1)
    model.to(device)

    for itr in range(iters):
        S = (S_ * mask)
        S = S + S.T
        A_prime = A_ + (S * C_)
        adj_hat = normalize_adj_tensor(A_prime + torch.eye(n_node,device=device))

        z1 = model(x_1, adj_1)
        z2 = model(x_2 + delta, adj_hat) 
        Attackloss = model.loss(z1, z2, batch_size=0) 
        Attackloss.backward()

        S_.data = (S_.data + alpha/np.sqrt(itr+1)*S_.grad.detach()) # annealing
        S_.data = bisection(S_.data, eps, xi) # clip S
        S_.grad.zero_()
        
        delta.data = (delta.data + beta*delta.grad.detach().sign()).clamp(-0.04,0.04)        
        delta.grad.zero_()

        randm = torch.rand(n_node, n_node,device=device)
        discretized_S = torch.where(S_.detach() > randm, torch.ones(n_node, n_node,device=device), torch.zeros(n_node, n_node, device=device))
        discretized_S = discretized_S + discretized_S.T
        A_hat = A_ + discretized_S * C_ + torch.eye(n_node,device=device)
        
    for param in model.parameters():
        param.requires_grad = True
    
    model.train()
    x_hat = x_2 + delta.data.to(device)
    assert torch.equal(A_hat, A_hat.transpose(0,1))
    return normalize_adj_tensor(A_hat), x_hat

def bisection(a,eps,xi,ub=1):
    pa = torch.clamp(a, 0, ub)
    if torch.sum(pa) <= eps:
        upper_S_update = pa
    else:
        mu_l = torch.min(a-1)
        mu_u = torch.max(a)
        mu_a = (mu_u + mu_l)/2
        while torch.abs(mu_u - mu_l)>xi:
            mu_a = (mu_u + mu_l)/2
            gu = torch.sum(torch.clamp(a-mu_a, 0, ub)) - eps
            gu_l = torch.sum(torch.clamp(a-mu_l, 0, ub)) - eps
            if gu == 0:
                break
            if torch.sign(gu) == torch.sign(gu_l):
                mu_l = mu_a
            else:
                mu_u = mu_a
        upper_S_update = torch.clamp(a-mu_a, 0, ub)
    return upper_S_update

## 1.4 Train function

In [ ]:
from time import perf_counter as t
from sklearn.decomposition import PCA
from topology_signature import TopologySignatureLoss

topo_loss_fn = TopologySignatureLoss(max_dimension=MAX_TOPOLOGY_DIM, loss_type=loss_type, distance_type=distance_type)

def Train(scAGCLmodel, data, cellGgraph, device, num_epochs, lam, alpha, beta, iters, optimizer, edge_r1, edge_r2, feature_r1, feature_r2, subgraph_size, zinb_loss_weight=1, topo_loss_weight=0, *, restore_best=False):
    start_time = t()
    scAGCLmodel.train()   
   
    Z_final=None
    best_loss = float("inf")
    best_state = None

    for epoch in range(num_epochs):
        # 1. Compute the Adversarial mode first
        scAGCLmodel.eval()
        X=data.x
        Adj = edge2adj(X, data.edge_index)
        X = X.to(device)
        Adj = Adj.to(device)
        with torch.no_grad():
            Z_current = scAGCLmodel(X, Adj)

        # Stage 2: attack the graph and use the augmented graph to train
        scAGCLmodel.train() #back to continue training

        subGraph = cellGgraph.subgraph(np.random.permutation(cellGgraph.number_of_nodes())[:subgraph_size])
        x_sub = data.x[np.array(subGraph.nodes())].to(device)
        
        subGraph = nx.relabel.convert_node_labels_to_integers(subGraph, first_label=0, ordering='default')      
        edgeind = np.asarray(list(subGraph.edges()), dtype=np.int64).reshape(-1, 2).T
        edgeind = torch.LongTensor(np.hstack([edgeind,edgeind[::-1]])).to(device)

        optimizer.zero_grad()
        x_1 = GeneDropping(x_sub, feature_r1)
        x_2 = GeneDropping(x_sub, feature_r2)

        edgeind_1 = EdgeDropping(edgeind, p=edge_r1, force_undirected=True)[0]
        edgeind_2 = EdgeDropping(edgeind, p=edge_r2, force_undirected=True)[0]
        adj_1 = edge2adj(x_1, edgeind_1)
        adj_2 = edge2adj(x_2, edgeind_2)

        z_1 = scAGCLmodel(x_1, adj_1)
        z_2 = scAGCLmodel(x_2, adj_2)    
        loss1= scAGCLmodel.loss(z_1,z_2,batch_size=0)
        
        loss2=0
        if lam > 0:
            adj_3, x_3 = GraphAdversarialAttack(scAGCLmodel, edgeind, edgeind_1, x_sub, x_1, iters, 0.2, alpha, beta)
            adj_4, x_4 = GraphAdversarialAttack(scAGCLmodel, edgeind, edgeind_2, x_sub, x_2, iters, 0.2, alpha, beta)
            z_3 = scAGCLmodel(x_3,adj_3)
            z_4 = scAGCLmodel(x_4,adj_4)
            loss2 = lam*scAGCLmodel.loss(z_3,z_4,batch_size=0)

        # Including loss of topology ==> Align z_1 with z_2 and z_3 with z_4
        loss3 = 0
        if topo_loss_weight > 0:
            loss3 = topo_loss_fn.forward_by_batch(z_1, z_2, batch_size=64)
            if lam > 0:
                loss3 = loss3 + topo_loss_fn.forward_by_batch(z_3, z_4, batch_size=64)
            loss3 = topo_loss_weight * loss3

        loss = loss1 + loss2 + loss3

        loss_value = loss.item()
        if not math.isfinite(loss_value):
            raise FloatingPointError(f"Non-finite loss at epoch {epoch}: {loss_value}")

        if loss_value < best_loss:
            best_loss = loss_value
            Z_final = Z_current
            if restore_best:
                best_state = {name: value.detach().clone() for name, value in scAGCLmodel.state_dict().items()}

        loss.backward()
        optimizer.step()

        if epoch % 50 == 0:
            now_time = t()
            print(f'Epoch={epoch:03d}, loss1={loss1:.4f}, loss2={loss2:.4f}, loss3={loss3:.4f}, total loss={loss:.4f}, total time {now_time - start_time:.4f}')

    if best_state is not None:
        scAGCLmodel.load_state_dict(best_state)
    return Z_final

# 2. Training Model

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import normalize as sklearn_normalize


def prepare_dataset(dataset):
    features, labels = load_h5_data1(BASE_DIR / f"{dataset}.h5")
    if len(labels) < 2:
        raise ValueError("At least two cells are required after preprocessing.")
    num_clusters = np.unique(labels).size
    data = Data(x=torch.from_numpy(features))
    cell_graph = GraphConstruction(data, features, num_clusters)
    return data, cell_graph, labels, num_clusters


def train_and_evaluate(prepared, seed, weight, keep_model):
    data, cell_graph, labels, num_clusters = prepared
    encoder = Encoder(dim_in=data.num_features, dim_out=num_hidden, num_layers=num_layers)
    model = AGCLModel(encoder, num_hidden, num_proj_hidden, tau).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    train_options = dict(
        lam=lam, alpha=alpha, beta=beta, iters=num_itersAdv, optimizer=optimizer,
        edge_r1=edge_r1, edge_r2=edge_r2, feature_r1=feature_r1, feature_r2=feature_r2,
        subgraph_size=subgraph_size, zinb_loss_weight=0,
    )

    if weight and num_epochs > topology_epochs:
        Train(
            model, data, cell_graph, device,
            num_epochs=num_epochs - topology_epochs, topo_loss_weight=0, **train_options,
        )
    embedding = Train(
        model, data, cell_graph, device,
        num_epochs=topology_epochs if weight else num_epochs,
        topo_loss_weight=weight, restore_best=keep_model, **train_options,
    )
    features = sklearn_normalize(embedding.detach().cpu().numpy(), norm="l2")
    predictions = KMeans(
        n_clusters=num_clusters, init="k-means++", random_state=seed
    ).fit_predict(features)
    return (
        normalized_mutual_info_score(labels, predictions),
        adjusted_rand_score(labels, predictions),
        model if keep_model else None,
    )

In [ ]:
TRAINING_MODES = {"Without topology": 0, "With topology": TOPO_WEIGHT}

if RUN_ROBUSTNESS and ROBUSTNESS_DATASET not in DATASETS:
    raise ValueError("ROBUSTNESS_DATASET must be included in DATASETS.")
if not 1 <= topology_epochs <= num_epochs or TOPO_WEIGHT <= 0:
    raise ValueError("Use a positive topology loss weight and 1 <= topology_epochs <= num_epochs.")

def run_experiments():
    scores = []
    selected_pair = None
    for dataset in DATASETS:
        print(f"\nDataset: {dataset}", flush=True)
        try:
            prepared = prepare_dataset(dataset)
        except Exception as error:
            print(f"{dataset}: preparation failed ({type(error).__name__}: {error})", flush=True)
            continue

        for seed in SEEDS:
            keep_models = RUN_ROBUSTNESS and dataset == ROBUSTNESS_DATASET and selected_pair is None
            models = []
            for mode, weight in TRAINING_MODES.items():
                run_label = f"{dataset} | seed={seed} | {mode}"
                print(run_label, flush=True)
                try:
                    random.seed(seed)
                    np.random.seed(seed)
                    torch.manual_seed(seed)
                    nmi, ari, model = train_and_evaluate(prepared, seed, weight, keep_models)
                    if not np.isfinite([nmi, ari]).all():
                        raise FloatingPointError("Non-finite clustering score.")
                except Exception as error:
                    print(f"{run_label}: failed ({type(error).__name__}: {error})", flush=True)
                    continue

                scores.append((dataset, seed, mode, nmi, ari))
                if keep_models:
                    models.append(model)

            if keep_models and len(models) == len(TRAINING_MODES):
                selected_pair = (prepared, models)
                print(f"Using {dataset} | seed={seed} for robustness evaluation.", flush=True)


    results = pd.DataFrame(
        scores, columns=["Dataset", "Seed", "Mode", "NMI", "ARI"]
    ).astype({"NMI": "float64", "ARI": "float64"})
    return results, selected_pair


run_results, robustness_pair = run_experiments()

## NMI and ARI by dataset

Mean and sample standard deviation (SD, ddof=1) are shown when every configured seed succeeds for that dataset and mode.

In [ ]:
summary = run_results.groupby(["Dataset", "Mode"]).agg(
    completed_seeds=("Seed", "count"),
    NMI_mean=("NMI", "mean"),
    NMI_SD=("NMI", "std"),
    ARI_mean=("ARI", "mean"),
    ARI_SD=("ARI", "std"),
)
expected_rows = pd.MultiIndex.from_product(
    [DATASETS, TRAINING_MODES], names=["Dataset", "Mode"]
)
summary = summary.reindex(expected_rows).rename(columns={
    "completed_seeds": "Completed seeds",
    "NMI_mean": "NMI mean",
    "NMI_SD": "NMI SD",
    "ARI_mean": "ARI mean",
    "ARI_SD": "ARI SD",
})
summary["Completed seeds"] = summary["Completed seeds"].fillna(0).astype(int)

incomplete = summary["Completed seeds"] != len(SEEDS)
metric_columns = ["NMI mean", "NMI SD", "ARI mean", "ARI SD"]
summary.loc[incomplete, metric_columns] = np.nan
display(summary.reset_index().round(4))

## Optional robustness evaluation

Set RUN_ROBUSTNESS to True and choose a ROBUSTNESS_DATASET from DATASETS.
The robustness evaluation uses the first seed that completes both training modes for that dataset.

In [ ]:
if RUN_ROBUSTNESS:
    from utils.robustness_topology import (
        DEFAULT_LEVELS, DEFAULT_SEEDS, make_default_appliers, run_robustness_comparison,
    )
    from utils.robustness_clustering import run_clustering_robustness_comparison

    ROBUSTNESS_LEVELS = DEFAULT_LEVELS
    ROBUSTNESS_SEEDS = DEFAULT_SEEDS
    # For a quick smoke run: ROBUSTNESS_LEVELS = (0.0, 0.1); ROBUSTNESS_SEEDS = (33333,)
    ROBUSTNESS_OUTPUT_DIR = PROJECT_ROOT / "results" / "robustness"


    if robustness_pair is None:
        raise RuntimeError(f"No seed completed both modes for {ROBUSTNESS_DATASET}.")
    prepared, fitted_models = robustness_pair

    clean_graph, _, Y, _ = prepared
    base_model, topology_model = fitted_models

    def make_get_representation_fn(model):
        def encode(x, edge_index):
            model.eval()
            model_device = next(model.parameters()).device
            x = torch.as_tensor(x, dtype=torch.float32, device=model_device)
            edge_index = edge_index.to(model_device)
            with torch.no_grad():
                z = model(x, edge2adj(x, edge_index))
            return z.detach().cpu()
        return encode

    pert_list = ["feature_dropout", "gaussian_noise", "edge_dropout"]
    ROBUSTNESS_APPLIERS = make_default_appliers(pert_list)
    models_for_robustness = {
        label: dict(get_representation_fn=make_get_representation_fn(model),
                    X_clean=clean_graph.x.detach().clone(),
                    aux_clean=clean_graph.edge_index.detach().clone(),
                    perturbation_appliers=ROBUSTNESS_APPLIERS)
        for label, model in [("Without Topology", base_model), ("With Topology", topology_model)]
    }
    robustness_labels = np.asarray(Y).reshape(-1)

    robustness_num_clusters = len(np.unique(robustness_labels))
    robustness_dataset = ROBUSTNESS_DATASET
    robustness_file_prefix = "scagcl" + "_" + Path(robustness_dataset).name

In [ ]:
if RUN_ROBUSTNESS:
    topology_results = run_robustness_comparison(
        models=models_for_robustness, dataset_name=robustness_dataset,
        perturbations=pert_list, levels=ROBUSTNESS_LEVELS, seeds=ROBUSTNESS_SEEDS,
        save_path=ROBUSTNESS_OUTPUT_DIR / (robustness_file_prefix + "_topology.csv"),
    )
    display(topology_results.head())

In [ ]:
if RUN_ROBUSTNESS:
    clustering_results = run_clustering_robustness_comparison(
        models=models_for_robustness, dataset_name=robustness_dataset,
        Y_true=robustness_labels, num_clusters=robustness_num_clusters,
        perturbations=pert_list, levels=ROBUSTNESS_LEVELS, seeds=ROBUSTNESS_SEEDS,
        save_path=ROBUSTNESS_OUTPUT_DIR / (robustness_file_prefix + "_clustering.csv"),
    )
    display(clustering_results.head())